# Phase 1: Data Integration and Feature Engineering

## Forecasting Competitive Football:
### Pre-Match and In-Play Prediction from Raw Event Data

This notebook implements the Phase 1 data engineering pipeline for the football forecasting project.

The objective of this phase is to transform raw football data sources into a validated, machine-learning-ready dataset while ensuring that all processing remains leakage-safe and reproducible.

The final outputs of this notebook are:

- An integrated match-level dataset.
- Fair bookmaker probability baselines.
- Classification and regression targets.
- Leakage-safe train/test splits.
- Initial pre-match features.
- A prototype in-play snapshot pipeline.

These outputs form the foundation for all predictive models developed in later phases.

# Project Objectives

The overall project consists of three forecasting tasks:

## Model 1: Pre-Match Outcome Classification

Predict:

- Home Win
- Draw
- Away Win

using only information available before kick-off.

---

## Model 2: Pre-Match Goal Margin Regression

Predict:

Home Goals − Away Goals

using the same pre-match feature set.

---

## Model 3: In-Play Prediction

Update match forecasts during a live match using:

- Current score
- Event stream information
- Match momentum
- Time remaining

while ensuring that no future events are visible at prediction time.

Phase 1 focuses exclusively on constructing the data infrastructure required for these future models.

In [2]:
import os
import random
import warnings
import pandas as pd
import numpy as np
from statsbombpy import sb

# -------------------------------------------------------------------------
# GLOBAL CONFIGURATION MATRIX
# -------------------------------------------------------------------------
CONFIG = {
    "SEED": 42,
    "COMP_NAME": "La Liga",
    "SEASON_NAME": "2015/2016",
    "ROLLING_WINDOW": 5,
    # Rolling window simulates information available before kickoff.
    "TRAIN_FRACTION": 0.80,
    "TARGET_SNAPSHOT_MIN": 30,
    "ODDS_PATH": "../data/raw/odds/spain_la_liga_2015_2016.csv"
}

# -------------------------------------------------------------------------
# ENVIRONMENT & REPRODUCIBILITY SEEDS
# -------------------------------------------------------------------------
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

random.seed(CONFIG["SEED"])
np.random.seed(CONFIG["SEED"])
os.environ["PYTHONHASHSEED"] = str(CONFIG["SEED"])

print(f"Pipeline initialized with fixed Random Seed: {CONFIG['SEED']}")
print(f"Target Sandbox: {CONFIG['COMP_NAME']} | Season: {CONFIG['SEASON_NAME']}")

Pipeline initialized with fixed Random Seed: 42
Target Sandbox: La Liga | Season: 2015/2016


# Data Sources

This project integrates two independent football data sources.

## 1. StatsBomb Open Data

StatsBomb provides:

- Competitions
- Seasons
- Matches
- Events
- Lineups
- 360 freeze-frame data

The dataset contains detailed event-level football information, including passes, shots, carries, pressures, fouls, duels, and goalkeeper actions.

---

## 2. Football-Data.co.uk

Football-Data provides:

- Historical bookmaker odds
- Match statistics
- Full-time results

These odds are used to construct the market benchmark against which future machine-learning models will be evaluated.

The integration of these two sources is a major objective of Phase 1.

# Competition Discovery

StatsBomb stores competitions and seasons separately.

Rather than hard-coding a competition identifier, the pipeline first queries all available competitions and seasons.

This approach:

- Improves reproducibility.
- Avoids hard-coded identifiers.
- Allows future scaling to additional leagues and seasons.

The selected competition for Phase 1 is:

- La Liga
- Season 2015/16

This season was chosen because it contains a complete league campaign with 380 matches and provides sufficient historical depth for feature engineering.

In [3]:
# -------------------------------------------------------------------------
# COMPETITION AND SEASON DISCOVERY
# -------------------------------------------------------------------------

# Retrieve every competition-season entry available in StatsBomb Open Data
competitions_df = sb.competitions()

required_competition_columns = {
    "competition_id",
    "season_id",
    "competition_name",
    "season_name",
}

missing_competition_columns = (
    required_competition_columns - set(competitions_df.columns)
)

if missing_competition_columns:
    raise ValueError(
        "Required competition columns are missing: "
        f"{sorted(missing_competition_columns)}"
    )

# Find the exact competition-season requested in CONFIG
selected_competition_df = competitions_df[
    (competitions_df["competition_name"] == CONFIG["COMP_NAME"])
    & (competitions_df["season_name"] == CONFIG["SEASON_NAME"])
].copy()

if selected_competition_df.empty:
    available_seasons = (
        competitions_df.loc[
            competitions_df["competition_name"] == CONFIG["COMP_NAME"],
            "season_name",
        ]
        .dropna()
        .sort_values()
        .unique()
        .tolist()
    )

    raise ValueError(
        f"No StatsBomb competition-season entry matched "
        f"{CONFIG['COMP_NAME']} | {CONFIG['SEASON_NAME']}. "
        f"Available seasons for this competition: {available_seasons}"
    )

if len(selected_competition_df) != 1:
    raise ValueError(
        "Competition discovery returned more than one matching row. "
        "The selected competition-season must be unique."
    )

selected_competition = selected_competition_df.iloc[0]

COMP_ID = int(selected_competition["competition_id"])
SEASON_ID = int(selected_competition["season_id"])

print("Competition-season discovery complete:")
print(f"   Competition: {CONFIG['COMP_NAME']}")
print(f"   Season: {CONFIG['SEASON_NAME']}")
print(f"   competition_id: {COMP_ID}")
print(f"   season_id: {SEASON_ID}")

display(
    selected_competition_df[
        [
            "competition_id",
            "season_id",
            "competition_name",
            "season_name",
        ]
    ]
)

Competition-season discovery complete:
   Competition: La Liga
   Season: 2015/2016
   competition_id: 11
   season_id: 27


,competition_id,season_id,competition_name,season_name
45,11,27,La Liga,2015/2016


In [4]:
# Pull down the complete master schedule table for the selected season configuration
raw_matches_df = sb.matches(
    competition_id=COMP_ID,
    season_id=SEASON_ID
)

print(
    f"Successfully parsed {len(raw_matches_df)} "
    f"match records from StatsBomb."
)

# Chronologically sort the master schedule right at ingestion to build a clean timeline
raw_matches_df["clean_date"] = pd.to_datetime(
    raw_matches_df["match_date"],
    errors="raise"
).dt.date

raw_matches_df = raw_matches_df.sort_values(
    by=["clean_date", "match_id"]
).reset_index(drop=True)

# Validate the primary match identifier before performing later relational joins
assert raw_matches_df["match_id"].notna().all(), \
    "Missing StatsBomb match_id values detected."

assert raw_matches_df["match_id"].is_unique, \
    "Duplicate StatsBomb match_id values detected."

# Isolate a single sample match record to validate event log parsers
sample_match_record = raw_matches_df.iloc[0]
SAMPLE_MATCH_ID = int(sample_match_record["match_id"])

print(
    f"Sandbox Target Match: "
    f"{sample_match_record['home_team']} vs "
    f"{sample_match_record['away_team']}"
)

# Extract the full event log block for this match record
# These events include passes, shots, carries, fouls, substitutions, and other actions
raw_events_df = sb.events(match_id=SAMPLE_MATCH_ID)

print(
    f"Ingested {len(raw_events_df)} raw events "
    f"for Match ID {SAMPLE_MATCH_ID}."
)

Successfully parsed 380 match records from StatsBomb.
Sandbox Target Match: Málaga vs Sevilla
Ingested 2750 raw events for Match ID 3825562.


# Match-Level Data Ingestion

The match table provides the fundamental unit of analysis for the pre-match models.

Each row represents a completed football match and contains:

- Match identifiers
- Teams
- Match date
- Final score
- Competition metadata

These records later become the basis for:

- Outcome labels
- Goal-margin labels
- Historical team statistics
- Train/test splits

# Event-Level Data Ingestion

Football matches are represented by ordered event streams.

Events include:

- Passes
- Carries
- Shots
- Pressures
- Fouls
- Goalkeeper actions

Although Phase 1 does not yet train predictive models from events, validating event ingestion is essential because future feature engineering and in-play prediction depend on event-level information.

A single match is therefore loaded as a proof-of-concept to verify event accessibility and schema consistency.

# Market Baseline Construction

StatsBomb does not contain betting-market information.

To construct a realistic benchmark, bookmaker odds from Football-Data.co.uk are integrated into the dataset.

The integration process consists of:

1. Team-name reconciliation.
2. Match tagging.
3. Coverage validation.
4. Probability conversion.
5. Vig removal.

This produces a fair market probability baseline that serves as the primary benchmark for future classification models.

# Team Name Reconciliation

StatsBomb and Football-Data use different naming conventions.

Examples include:

- Atlético Madrid vs Atletico Madrid
- Deportivo La Coruña vs La Coruna

Because no shared match identifier exists across sources, successful integration requires explicit alias handling.

The mapping process ensures that equivalent teams are matched consistently across both datasets.

# Fair Probability Estimation

Bookmaker odds include an over-round (vig), meaning the implied probabilities sum to more than one.

To recover fair probabilities:

1. Decimal odds are converted into implied probabilities.
2. The three probabilities are normalised.
3. The resulting probabilities sum exactly to one.

These de-vigged probabilities represent the bookmaker's true forecast and will serve as the market benchmark throughout the project.

In [5]:
# Master translation matrix mapping StatsBomb team strings to Football-Data CSV tokens
# StatsBomb and Football-Data do not share a match ID. The notebook creates a deterministic team-name alias map
# The sources use different names.
TEAM_ALIAS_MAP = {
    "Athletic Club": "Ath Bilbao",
    "Atlético Madrid": "Ath Madrid",
    "Celta Vigo": "Celta",
    "RC Deportivo La Coruña": "La Coruna",
    "Eibar": "Eibar",
    "Espanyol": "Espanol",
    "FC Barcelona": "Barcelona",
    "Barcelona": "Barcelona",
    "Getafe": "Getafe",
    "Granada": "Granada",
    "Las Palmas": "Las Palmas",
    "Levante UD": "Levante",
    "Málaga": "Malaga",
    "Rayo Vallecano": "Vallecano",
    "Real Betis": "Betis",
    "Real Madrid": "Real Madrid",
    "Real Sociedad": "Sociedad",
    "Sevilla": "Sevilla",
    "Sporting Gijón": "Sp Gijon",
    "Valencia": "Valencia",
    "Villarreal": "Villarreal"
}

try:
    # Read local historical betting odds CSV file
    market_odds_df = pd.read_csv(CONFIG["ODDS_PATH"])

    required_odds_columns = {
        "Date",
        "HomeTeam",
        "AwayTeam",
        "B365H",
        "B365D",
        "B365A"
    }

    missing_odds_columns = (
        required_odds_columns - set(market_odds_df.columns)
    )

    if missing_odds_columns:
        raise ValueError(
            f"Required odds columns are missing: "
            f"{sorted(missing_odds_columns)}"
        )

    # Clean and parse the odds file date column to align format indices
    market_odds_df["clean_date"] = pd.to_datetime(
        market_odds_df["Date"],
        format="%d/%m/%y",
        errors="raise"
    ).dt.date

    # Map raw structural names to alignment tokens
    raw_matches_df["market_home"] = raw_matches_df["home_team"].map(
        TEAM_ALIAS_MAP
    )
    raw_matches_df["market_away"] = raw_matches_df["away_team"].map(
        TEAM_ALIAS_MAP
    )

    # Ensure that every StatsBomb team has an explicit alias before merging
    missing_home_aliases = raw_matches_df.loc[
        raw_matches_df["market_home"].isna(),
        "home_team"
    ].unique()

    missing_away_aliases = raw_matches_df.loc[
        raw_matches_df["market_away"].isna(),
        "away_team"
    ].unique()

    missing_aliases = sorted(
        set(missing_home_aliases) | set(missing_away_aliases)
    )

    if missing_aliases:
        raise ValueError(
            f"Missing team aliases for: {missing_aliases}"
        )

    # Verify that Football-Data does not contain duplicate scheduling identities
    duplicate_odds_keys = market_odds_df.duplicated(
        subset=["clean_date", "HomeTeam", "AwayTeam"],
        keep=False
    )

    if duplicate_odds_keys.any():
        duplicate_rows = market_odds_df.loc[
            duplicate_odds_keys,
            ["clean_date", "HomeTeam", "AwayTeam"]
        ]

        raise ValueError(
            "Duplicate Football-Data scheduling keys detected:\n"
            f"{duplicate_rows.to_string(index=False)}"
        )

    # Relational join utilizing strict scheduling attributes
    odds_target_cols = [
        "clean_date",
        "HomeTeam",
        "AwayTeam",
        "B365H",
        "B365D",
        "B365A"
    ]

    integrated_matches_df = pd.merge(
        raw_matches_df,
        market_odds_df[odds_target_cols],
        left_on=["clean_date", "market_home", "market_away"],
        right_on=["clean_date", "HomeTeam", "AwayTeam"],
        how="left",
        validate="one_to_one",
        indicator=True
    ).sort_values(
        by=["clean_date", "match_id"]
    ).reset_index(drop=True)

    # Identify and retain evidence for every failed odds match
    unmatched_matches_df = integrated_matches_df[
        integrated_matches_df["_merge"] != "both"
    ][
        [
            "match_id",
            "clean_date",
            "home_team",
            "away_team",
            "market_home",
            "market_away"
        ]
    ].copy()

    matched_count = (
        integrated_matches_df["_merge"] == "both"
    ).sum()

    # Fulfill reporting requirement: Compute and print verification coverage
    # 100% means every StatsBomb fixture in the season was matched to a Football-Data row.
    coverage_rate = (
        matched_count / len(raw_matches_df)
    ) * 100

    print(
        f"Relational Tagging Coverage Rate: "
        f"{coverage_rate:.2f}% "
        f"({matched_count}/{len(raw_matches_df)} Matches)"
    )

    if not unmatched_matches_df.empty:
        print("\nUnmatched fixtures:")
        print(unmatched_matches_df.to_string(index=False))

    # Keep only successfully tagged matches for market-comparison experiments
    integrated_matches_df = integrated_matches_df[
        integrated_matches_df["_merge"] == "both"
    ].drop(columns="_merge").reset_index(drop=True)

    # Validate the decimal odds before calculating inverse probabilities
    odds_columns = ["B365H", "B365D", "B365A"]

    if integrated_matches_df[odds_columns].isna().any().any():
        raise ValueError(
            "Missing bookmaker odds detected after integration."
        )

    if (integrated_matches_df[odds_columns] <= 1.0).any().any():
        raise ValueError(
            "Invalid decimal odds detected. Decimal odds must exceed 1.0."
        )

    # Formulate de-vigged fair market probabilities (1 / Decimal Odd)
    integrated_matches_df["implied_H"] = (
        1 / integrated_matches_df["B365H"]
    )
    integrated_matches_df["implied_D"] = (
        1 / integrated_matches_df["B365D"]
    )
    integrated_matches_df["implied_A"] = (
        1 / integrated_matches_df["B365A"]
    )

    # Calculate the total over-round margin
    integrated_matches_df["bookie_margin"] = (
        integrated_matches_df["implied_H"]
        + integrated_matches_df["implied_D"]
        + integrated_matches_df["implied_A"]
    )

    assert (
        integrated_matches_df["bookie_margin"] > 0
    ).all(), "Non-positive bookmaker margin detected."

    # Normalize probabilities to sum to exactly 1.0
    integrated_matches_df["market_prob_H"] = (
        integrated_matches_df["implied_H"]
        / integrated_matches_df["bookie_margin"]
    )

    integrated_matches_df["market_prob_D"] = (
        integrated_matches_df["implied_D"]
        / integrated_matches_df["bookie_margin"]
    )

    integrated_matches_df["market_prob_A"] = (
        integrated_matches_df["implied_A"]
        / integrated_matches_df["bookie_margin"]
    )

    # Confirm that every de-vigged probability triplet forms a valid distribution
    market_probability_sum = integrated_matches_df[
        ["market_prob_H", "market_prob_D", "market_prob_A"]
    ].sum(axis=1)

    assert np.allclose(
        market_probability_sum.to_numpy(),
        1.0,
        atol=1e-10
    ), "De-vigged market probabilities do not sum to 1.0."

    assert integrated_matches_df[
        ["market_prob_H", "market_prob_D", "market_prob_A"]
    ].apply(
        lambda column: column.between(0, 1).all()
    ).all(), "Invalid market probability outside [0, 1]."

    print(
        "Fair Market Probability Baseline established successfully."
    )
    print(
        "Probability-sum assertion passed for every tagged match."
    )

except FileNotFoundError:
    raise FileNotFoundError(
        f"Integration Error: File missing at designated path: "
        f"{CONFIG['ODDS_PATH']}"
    )

Relational Tagging Coverage Rate: 100.00% (380/380 Matches)
Fair Market Probability Baseline established successfully.
Probability-sum assertion passed for every tagged match.


# Leakage-Safe Train/Test Split

Football forecasting is a temporal prediction problem.

Future matches must never influence training data.

Therefore:

- Training matches occur strictly before test matches.
- No date overlap is permitted.
- The split is performed chronologically rather than randomly.

This design mirrors real-world deployment and prevents temporal leakage.

In [6]:
# Calculate the approximate split index cleanly based on sequential rows
approximate_split_index = int(
    len(integrated_matches_df) * CONFIG["TRAIN_FRACTION"]
)

if approximate_split_index <= 0 or approximate_split_index >= len(
    integrated_matches_df
):
    raise ValueError(
        "TRAIN_FRACTION produces an empty training or test partition."
    )

# Select the first calendar date assigned fully to the held-out test set
split_date = integrated_matches_df.iloc[
    approximate_split_index
]["clean_date"]

# Slice dataframes to establish a clean chronological barrier
# Every match on the split date is assigned to the test partition.
train_val_pool = integrated_matches_df[
    integrated_matches_df["clean_date"] < split_date
].copy()

test_pool = integrated_matches_df[
    integrated_matches_df["clean_date"] >= split_date
].copy()

# Confirm that the chronological boundary is strict and has no date overlap
assert not train_val_pool.empty, \
    "Training/validation partition is empty."

assert not test_pool.empty, \
    "Test partition is empty."

assert (
    train_val_pool["clean_date"].max()
    < test_pool["clean_date"].min()
), (
    "Temporal split failure: training and test date ranges overlap."
)

assert set(train_val_pool["match_id"]).isdisjoint(
    set(test_pool["match_id"])
), "The same match_id appears in both train and test partitions."

print("Temporal Partition Complete:")
print(f"   ├── Split Date               : {split_date}")
print(
    f"   ├── Training History Window  : "
    f"{len(train_val_pool)} Matches "
    f"({train_val_pool['clean_date'].min()} "
    f"to {train_val_pool['clean_date'].max()})"
)
print(
    f"   └── Test Evaluation Window   : "
    f"{len(test_pool)} Matches "
    f"({test_pool['clean_date'].min()} "
    f"to {test_pool['clean_date'].max()})"
)

Temporal Partition Complete:
   ├── Split Date               : 2016-04-02
   ├── Training History Window  : 301 Matches (2015-08-21 to 2016-04-01)
   └── Test Evaluation Window   : 79 Matches (2016-04-02 to 2016-05-15)


# Target Construction

Machine-learning models require clearly defined prediction targets.

Two targets are created:

## Classification Target

Predict:

- Home Win
- Draw
- Away Win

## Regression Target

Predict:

Home Goals − Away Goals

The regression target is clipped to the interval [-5, 5] to reduce the influence of rare extreme scorelines.

# Leakage-Safe Feature Engineering

The first predictive features are derived from historical team performance.

For every target match, only matches completed before kick-off are used.

The feature set contains:

- Home rolling goals scored
- Home rolling goals conceded
- Away rolling goals scored
- Away rolling goals conceded
- Home history count
- Away history count

The history-count features quantify the amount of historical information available for each team and help distinguish early-season matches from later-season matches.

# Separation of Model Features and Market Baseline

The bookmaker probabilities are intentionally excluded from the machine-learning feature matrix.

Reason:

If bookmaker probabilities were used as model inputs while simultaneously serving as the evaluation benchmark, the comparison would become circular.

By excluding:

- market_prob_H
- market_prob_D
- market_prob_A

the project preserves a fair comparison between:

- Machine-learning models
- Bookmaker forecasts

In [7]:
def extract_ground_truth_labels(dataframe):
    """Derives signed goal differences and multi-class categorical match outcomes

    from final score metrics.
    """
    processed_df = dataframe.copy()

    required_score_columns = {"home_score", "away_score"}
    missing_score_columns = (
        required_score_columns - set(processed_df.columns)
    )

    if missing_score_columns:
        raise ValueError(
            f"Required score columns are missing: "
            f"{sorted(missing_score_columns)}"
        )

    if processed_df[
        ["home_score", "away_score"]
    ].isna().any().any():
        raise ValueError(
            "Missing final scores detected during target generation."
        )

    # Calculate the original signed goal margin before clipping
    raw_goal_margin = (
        processed_df["home_score"]
        - processed_df["away_score"]
    )

    # Model 2 Target: Home Score minus Away Score, clipped strictly to [-5, +5]
    processed_df["target_margin"] = raw_goal_margin.clip(-5, 5)

    # Model 1 Target: Multi-class label mapping (H=Home Win, D=Draw, A=Away Win)
    # Classification is derived from the original score margin rather than the clipped regression label.
    processed_df["target_outcome"] = np.select(
        [
            raw_goal_margin > 0,
            raw_goal_margin < 0
        ],
        [
            "H",
            "A"
        ],
        default="D"
    )

    assert processed_df["target_margin"].between(
        -5, 5
    ).all(), "Regression target exceeds the permitted [-5, +5] range."

    assert set(
        processed_df["target_outcome"].unique()
    ).issubset(
        {"H", "D", "A"}
    ), "Unexpected classification target detected."

    return processed_df


# Generate labels for both training and testing datasets
train_val_pool = extract_ground_truth_labels(train_val_pool)
test_pool = extract_ground_truth_labels(test_pool)

print("📊 Training Distribution Baseline Base-Rates:")
print(
    train_val_pool["target_outcome"]
    .value_counts(normalize=True)
    .sort_index()
)

📊 Training Distribution Baseline Base-Rates:
target_outcome
A    0.275748
D    0.249169
H    0.475083
Name: proportion, dtype: float64


In [8]:
def execute_leakage_proof_rolling_engine(
    target_scope,
    master_history_pool,
    rolling_window=5
):
    """Calculates rolling Goals For (GF) and Goals Against (GA) averages

    using only matches completed prior to the fixture date.
    """
    engineered_output = []

    if rolling_window <= 0:
        raise ValueError("rolling_window must be a positive integer.")

    for _, current_fixture in target_scope.iterrows():
        fixture_date = current_fixture["clean_date"]
        home_team = current_fixture["home_team"]
        away_team = current_fixture["away_team"]

        # FILTER: Exclude any matches played on or after the target fixture date
        # This strict inequality ensures that the current target match is excluded.
        past_historical_pool = master_history_pool[
            master_history_pool["clean_date"] < fixture_date
        ]

        # Assert that no future or same-date match enters the historical pool
        if not past_historical_pool.empty:
            assert (
                past_historical_pool["clean_date"].max()
                < fixture_date
            ), (
                f"Historical leakage detected for match "
                f"{current_fixture['match_id']}."
            )

        assert (
            current_fixture["match_id"]
            not in set(past_historical_pool["match_id"])
        ), (
            f"Target match {current_fixture['match_id']} "
            f"was found inside its own historical pool."
        )

        def calculate_rolling_metrics(team_name):
            # Isolate team matches regardless of whether they played home or away
            team_games = past_historical_pool[
                (past_historical_pool["home_team"] == team_name)
                |
                (past_historical_pool["away_team"] == team_name)
            ].sort_values(
                by=["clean_date", "match_id"],
                ascending=[False, False]
            )

            if len(team_games) == 0:
                return 0.0, 0.0, 0  # Default value for opening fixtures

            # Limit lookback to the configured window size
            # It locates the last five games for each team, regardless of whether the team was home or away.
            recent_sample = team_games.head(rolling_window)

            goals_scored = []
            goals_conceded = []

            # Then it calculates th average goal scored, and average goal conceded
            for _, game in recent_sample.iterrows():
                if game["home_team"] == team_name:
                    goals_scored.append(game["home_score"])
                    goals_conceded.append(game["away_score"])
                else:
                    goals_scored.append(game["away_score"])
                    goals_conceded.append(game["home_score"])

            return (
                float(np.mean(goals_scored)),
                float(np.mean(goals_conceded)),
                len(recent_sample)
            )

        # Calculate trailing averages for both teams
        home_gf, home_ga, home_history_count = (
            calculate_rolling_metrics(home_team)
        )

        away_gf, away_ga, away_history_count = (
            calculate_rolling_metrics(away_team)
        )

        # Assemble feature row record
        engineered_output.append({
            "match_id": current_fixture["match_id"],
            "clean_date": fixture_date,
            "home_team": home_team,
            "away_team": away_team,

            "feat_home_rolling_gf": home_gf,
            # Home team's average goals scored in its previous five matches

            "feat_home_rolling_ga": home_ga,
            # Home team's average goals conceded in its previous five matches

            "feat_away_rolling_gf": away_gf,
            # Away team's average goals scored in its previous five matches

            "feat_away_rolling_ga": away_ga,
            # Away team's average goals conceded in its previous five matches

            # History-count features distinguish missing history from a genuine zero average
            "feat_home_history_count": home_history_count,
            "feat_away_history_count": away_history_count,

            # Market columns are retained for baseline evaluation only.
            # They are not included in the main event-derived model feature list below.
            "market_prob_H": current_fixture["market_prob_H"],
            "market_prob_D": current_fixture["market_prob_D"],
            "market_prob_A": current_fixture["market_prob_A"],

            "target_margin": current_fixture["target_margin"],
            "target_outcome": current_fixture["target_outcome"]
        })

    engineered_df = pd.DataFrame(engineered_output)

    assert engineered_df["match_id"].is_unique, \
        "Duplicate match rows detected in the engineered matrix."

    return engineered_df


print("Processing Training Feature Matrix...")
X_y_train_val_matrix = execute_leakage_proof_rolling_engine(
    train_val_pool,
    integrated_matches_df,
    CONFIG["ROLLING_WINDOW"]
)

print("Processing Test Feature Matrix...")
X_y_test_matrix = execute_leakage_proof_rolling_engine(
    test_pool,
    integrated_matches_df,
    CONFIG["ROLLING_WINDOW"]
)

# -------------------------------------------------------------------------
# EXPLICIT EXPERIMENTAL COLUMN SEPARATION
# -------------------------------------------------------------------------
# These columns are the actual model inputs for the event-derived pre-match models.
MODEL_FEATURE_COLUMNS = [
    "feat_home_rolling_gf",
    "feat_home_rolling_ga",
    "feat_away_rolling_gf",
    "feat_away_rolling_ga",
    "feat_home_history_count",
    "feat_away_history_count"
]

# These columns form the external bookmaker baseline.
# Do not include them in MODEL_FEATURE_COLUMNS when comparing the model against the market.
MARKET_BASELINE_COLUMNS = [
    "market_prob_H",
    "market_prob_D",
    "market_prob_A"
]

CLASSIFICATION_TARGET_COLUMN = "target_outcome"
REGRESSION_TARGET_COLUMN = "target_margin"

# Classification model matrices
X_train_val = X_y_train_val_matrix[MODEL_FEATURE_COLUMNS].copy()
X_test = X_y_test_matrix[MODEL_FEATURE_COLUMNS].copy()

y_train_val_classification = X_y_train_val_matrix[
    CLASSIFICATION_TARGET_COLUMN
].copy()

y_test_classification = X_y_test_matrix[
    CLASSIFICATION_TARGET_COLUMN
].copy()

# Regression targets use the same pre-match features
y_train_val_regression = X_y_train_val_matrix[
    REGRESSION_TARGET_COLUMN
].copy()

y_test_regression = X_y_test_matrix[
    REGRESSION_TARGET_COLUMN
].copy()

# Market probabilities remain separate for fair baseline comparison
market_baseline_train_val = X_y_train_val_matrix[
    MARKET_BASELINE_COLUMNS
].copy()

market_baseline_test = X_y_test_matrix[
    MARKET_BASELINE_COLUMNS
].copy()

# Final structural assertions
assert set(MODEL_FEATURE_COLUMNS).isdisjoint(
    set(MARKET_BASELINE_COLUMNS)
), "Market baseline columns leaked into the main feature list."

assert len(X_train_val) == len(y_train_val_classification)
assert len(X_test) == len(y_test_classification)
assert len(X_train_val) == len(y_train_val_regression)
assert len(X_test) == len(y_test_regression)

assert np.allclose(
    market_baseline_test.sum(axis=1).to_numpy(),
    1.0,
    atol=1e-10
), "Test-set market probabilities do not sum to 1.0."

print("\nMatrix Processing Complete. Mid-Season Feature Verification Sample:")
print(
    X_y_train_val_matrix[
        [
            "clean_date",
            "home_team",
            "feat_home_rolling_gf",
            "feat_home_rolling_ga",
            "feat_home_history_count"
        ]
    ].iloc[100:103]
)

print("\nExperimental column separation verified:")
print(f"   ├── Model Features          : {MODEL_FEATURE_COLUMNS}")
print(f"   ├── Market Baseline Columns : {MARKET_BASELINE_COLUMNS}")
print(f"   ├── Training Matrix Shape   : {X_train_val.shape}")
print(f"   └── Test Matrix Shape       : {X_test.shape}")

Processing Training Feature Matrix...
Processing Test Feature Matrix...

Matrix Processing Complete. Mid-Season Feature Verification Sample:
     clean_date   home_team  feat_home_rolling_gf  feat_home_rolling_ga  \
100  2015-11-06  Las Palmas                   0.4                   2.2   
101  2015-11-07       Eibar                   1.2                   1.0   
102  2015-11-07  Celta Vigo                   1.4                   1.4   

     feat_home_history_count  
100                        5  
101                        5  
102                        5  

Experimental column separation verified:
   ├── Model Features          : ['feat_home_rolling_gf', 'feat_home_rolling_ga', 'feat_away_rolling_gf', 'feat_away_rolling_ga', 'feat_home_history_count', 'feat_away_history_count']
   ├── Market Baseline Columns : ['market_prob_H', 'market_prob_D', 'market_prob_A']
   ├── Training Matrix Shape   : (301, 6)
   └── Test Matrix Shape       : (79, 6)


# Prototype In-Play Snapshot Pipeline

The final component of Phase 1 is a prototype live-match pipeline.

A snapshot at minute t is created by:

1. Sorting events chronologically.
2. Retaining only events occurring at or before t.
3. Computing match-state summaries.

This prototype demonstrates that future in-play models can be constructed without violating the time boundary required for leakage-safe live prediction.

In [9]:
def generate_in_play_snapshot(
    match_event_stream,
    snapshot_minute_t
):
    """Filters a match event stream up to snapshot_minute_t

    and runs a strict assertion checking for look-ahead leakage.
    """
    if snapshot_minute_t < 0:
        raise ValueError(
            "snapshot_minute_t must be non-negative."
        )

    required_event_columns = {
        "minute",
        "type"
    }

    missing_event_columns = (
        required_event_columns - set(match_event_stream.columns)
    )

    if missing_event_columns:
        raise ValueError(
            f"Required event columns are missing: "
            f"{sorted(missing_event_columns)}"
        )

    events_pool = match_event_stream.copy()

    if events_pool["minute"].isna().any():
        raise ValueError(
            "Missing event-minute values detected."
        )

    # Sort using every available temporal ordering field
    temporal_sort_columns = [
        column
        for column in ["period", "timestamp", "index"]
        if column in events_pool.columns
    ]

    if temporal_sort_columns:
        events_pool = events_pool.sort_values(
            by=temporal_sort_columns
        ).reset_index(drop=True)

    # Apply strict temporal mask to capture events up to time t only
    snapshot_subset_df = events_pool[
        events_pool["minute"] <= snapshot_minute_t
    ].copy()

    future_events_df = events_pool[
        events_pool["minute"] > snapshot_minute_t
    ].copy()

    # -------------------------------------------------------------------------
    # MANDATORY RUNTIME TIME-T ASSERTION CONSTRAINT
    # -------------------------------------------------------------------------
    if not snapshot_subset_df.empty:
        max_detected_minute = snapshot_subset_df["minute"].max()

        assert max_detected_minute <= snapshot_minute_t, \
            (
                f"CRITICAL LEAKAGE DETECTED: Event minute "
                f"({max_detected_minute}) exceeds snapshot target "
                f"boundary ({snapshot_minute_t})."
            )

    if not future_events_df.empty:
        assert (
            future_events_df["minute"].min()
            > snapshot_minute_t
        ), (
            "Temporal partition failure: a future-event row does not "
            "strictly exceed the snapshot boundary."
        )

    assert len(snapshot_subset_df) + len(future_events_df) == len(
        events_pool
    ), "Event rows were lost or duplicated during snapshot partitioning."

    # Calculate granular feature metrics up to time t
    count_shots = (
        snapshot_subset_df["type"] == "Shot"
    ).sum()

    count_passes = (
        snapshot_subset_df["type"] == "Pass"
    ).sum()

    snapshot_payload = {
        "snapshot_minute_t": snapshot_minute_t,
        "events_accumulated": len(snapshot_subset_df),
        "feat_live_cumulative_shots": int(count_shots),
        "feat_live_cumulative_passes": int(count_passes)
    }

    return snapshot_payload, snapshot_subset_df


# Evaluate snapshot utility performance at the Minute 30 milestone
snapshot_metrics, snapshot_filtered_events = (
    generate_in_play_snapshot(
        raw_events_df,
        CONFIG["TARGET_SNAPSHOT_MIN"]
    )
)

print(
    f"Time-t Verification Log "
    f"(Minute {CONFIG['TARGET_SNAPSHOT_MIN']}):"
)

print(
    f"   ├── Retained Event Count: "
    f"{snapshot_metrics['events_accumulated']} Rows"
)

print(
    f"   ├── Cumulative Passes   : "
    f"{snapshot_metrics['feat_live_cumulative_passes']}"
)

print(
    f"   └── Cumulative Shots    : "
    f"{snapshot_metrics['feat_live_cumulative_shots']}"
)

print(
    "\nExecution verified as leakage-aware. "
    "Time-t boundary assertions passed successfully."
)

Time-t Verification Log (Minute 30):
   ├── Retained Event Count: 965 Rows
   ├── Cumulative Passes   : 278
   └── Cumulative Shots    : 14

Execution verified as leakage-aware. Time-t boundary assertions passed successfully.


# Phase 1 Summary

Phase 1 successfully transformed raw football data into a machine-learning-ready forecasting infrastructure.

Completed deliverables:

- Competition discovery
- Match ingestion
- Event ingestion
- Odds integration
- Team alias resolution
- Fair probability estimation
- Classification target generation
- Regression target generation
- Leakage-safe train/test split
- Leakage-safe feature engineering
- Prototype in-play snapshot generation

The resulting pipeline provides the foundation for Phase 2, where predictive models will be trained, evaluated, calibrated, and compared against the bookmaker market baseline.

In [12]:
# -------------------------------------------------------------------------
# EXPORT A CLEAN PHASE 1 MATCH TABLE
# -------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np

PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

MATCH_OUTPUT_PATH = (
    PROCESSED_DATA_DIR
    / "integrated_matches_laliga_2015_2016.parquet"
)

# -------------------------------------------------------------------------
# COLUMNS NEEDED BY LATER NOTEBOOKS
# -------------------------------------------------------------------------

preferred_export_columns = [
    # Core match identity
    "match_id",
    "clean_date",
    "home_team",
    "away_team",

    # Final scores
    "home_score",
    "away_score",

    # Market odds
    "odds_H",
    "odds_D",
    "odds_A",

    # Raw implied probabilities
    "implied_H",
    "implied_D",
    "implied_A",
    "bookie_margin",

    # De-vigged probabilities
    "market_prob_H",
    "market_prob_D",
    "market_prob_A",

    # Targets, if already created
    "raw_goal_margin",
    "target_classification",
    "target_regression",
]

# Keep only columns that currently exist.
export_columns = [
    column
    for column in preferred_export_columns
    if column in integrated_matches_df.columns
]

required_columns = {
    "match_id",
    "clean_date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
}

missing_required_columns = (
    required_columns - set(export_columns)
)

if missing_required_columns:
    raise ValueError(
        "Required match columns are missing: "
        f"{sorted(missing_required_columns)}"
    )

export_matches_df = integrated_matches_df[
    export_columns
].copy()

# -------------------------------------------------------------------------
# STANDARDISE DATA TYPES
# -------------------------------------------------------------------------

export_matches_df["match_id"] = pd.to_numeric(
    export_matches_df["match_id"],
    errors="raise"
).astype("int64")

export_matches_df["clean_date"] = pd.to_datetime(
    export_matches_df["clean_date"],
    errors="raise"
)

export_matches_df["home_team"] = (
    export_matches_df["home_team"]
    .astype("string")
)

export_matches_df["away_team"] = (
    export_matches_df["away_team"]
    .astype("string")
)

export_matches_df["home_score"] = pd.to_numeric(
    export_matches_df["home_score"],
    errors="raise"
).astype("int64")

export_matches_df["away_score"] = pd.to_numeric(
    export_matches_df["away_score"],
    errors="raise"
).astype("int64")

# Convert remaining numeric columns safely.
numeric_candidate_columns = [
    "odds_H",
    "odds_D",
    "odds_A",
    "implied_H",
    "implied_D",
    "implied_A",
    "bookie_margin",
    "market_prob_H",
    "market_prob_D",
    "market_prob_A",
    "raw_goal_margin",
    "target_regression",
]

for column in numeric_candidate_columns:
    if column in export_matches_df.columns:
        export_matches_df[column] = pd.to_numeric(
            export_matches_df[column],
            errors="coerce"
        )

# Classification target is categorical text.
if "target_classification" in export_matches_df.columns:
    export_matches_df["target_classification"] = (
        export_matches_df["target_classification"]
        .astype("string")
    )

# -------------------------------------------------------------------------
# VALIDATION
# -------------------------------------------------------------------------

if export_matches_df["match_id"].isna().any():
    raise ValueError("Missing match IDs detected.")

if not export_matches_df["match_id"].is_unique:
    raise ValueError("Duplicate match IDs detected.")

if export_matches_df["clean_date"].isna().any():
    raise ValueError("Missing match dates detected.")

if export_matches_df["home_team"].isna().any():
    raise ValueError("Missing home-team values detected.")

if export_matches_df["away_team"].isna().any():
    raise ValueError("Missing away-team values detected.")

# -------------------------------------------------------------------------
# EXPORT
# -------------------------------------------------------------------------

export_matches_df.to_parquet(
    MATCH_OUTPUT_PATH,
    index=False,
    engine="pyarrow"
)

print("Clean Phase 1 match table exported successfully.")
print(f"Path   : {MATCH_OUTPUT_PATH.resolve()}")
print(f"Rows   : {len(export_matches_df):,}")
print(f"Columns: {len(export_matches_df.columns):,}")
print("\nExported columns:")

for column in export_matches_df.columns:
    print(f"  - {column}: {export_matches_df[column].dtype}")

Clean Phase 1 match table exported successfully.
Path   : /home/sina/Documents/Arshad/Spring - 2026/Machine Learning/home works/Project/data/processed/integrated_matches_laliga_2015_2016.parquet
Rows   : 380
Columns: 13

Exported columns:
  - match_id: int64
  - clean_date: datetime64[ns]
  - home_team: string
  - away_team: string
  - home_score: int64
  - away_score: int64
  - implied_H: float64
  - implied_D: float64
  - implied_A: float64
  - bookie_margin: float64
  - market_prob_H: float64
  - market_prob_D: float64
  - market_prob_A: float64


In [13]:
# -------------------------------------------------------------------------
# RELOAD AND VALIDATE THE EXPORTED TABLE
# -------------------------------------------------------------------------

reloaded_matches_df = pd.read_parquet(MATCH_OUTPUT_PATH)

assert len(reloaded_matches_df) == len(export_matches_df)
assert reloaded_matches_df["match_id"].notna().all()
assert reloaded_matches_df["match_id"].is_unique

assert set(reloaded_matches_df["match_id"]) == set(
    export_matches_df["match_id"]
)

print("Reload validation passed.")
print(f"Validated matches: {len(reloaded_matches_df):,}")

display(reloaded_matches_df.head())

Reload validation passed.
Validated matches: 380


,match_id,clean_date,home_team,away_team,home_score,away_score,implied_H,implied_D,implied_A,bookie_margin,market_prob_H,market_prob_D,market_prob_A
0,3825562,2015-08-21,Málaga,Sevilla,0,0,0.294118,0.294118,0.465116,1.053352,0.279221,0.279221,0.441558
1,3825563,2015-08-22,Atlético Madrid,Las Palmas,1,0,0.833333,0.153846,0.066667,1.053846,0.790754,0.145985,0.063260
2,3825564,2015-08-22,RC Deportivo La Coruña,Real Sociedad,0,0,0.400000,0.312500,0.333333,1.045833,0.382470,0.298805,0.318725
3,3825565,2015-08-22,Espanyol,Getafe,1,0,0.523560,0.294118,0.230947,1.048625,0.499283,0.280479,0.220238
4,3825566,2015-08-22,Rayo Vallecano,Valencia,0,0,0.277778,0.285714,0.487805,1.051297,0.264224,0.271773,0.464003
